# PV Module Lifetime and Waste-Flow Forecasting

**Project walkthrough:** Weibull lifetime fitting with right-censored observations, annual cohort retirements and waste mass estimation.

> The included data are synthetic. Outputs are methodological demonstrations, not Bangladesh national PV waste estimates.

In [ ]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from pv_waste_forecasting.model import (
    fit_weibull_lifetime, survival_probability,
    forecast_cohort_waste
)

lifetime = pd.read_csv(ROOT / "data/demo/module_lifetime_observations_synthetic.csv")
cohorts = pd.read_csv(ROOT / "data/demo/pv_installation_cohorts_synthetic.csv")
settings = json.loads((ROOT / "config/base_case.json").read_text())

display(lifetime.head())
display(cohorts.head())

## 1. Fit a Weibull model with right-censored observations

In [ ]:
fit = fit_weibull_lifetime(
    lifetime["observed_lifetime_years"],
    lifetime["event_observed"]
)
fit

## 2. Survival curve and cohort retirement forecast

In [ ]:
ages = np.linspace(0, 45, 200)
survival = survival_probability(ages, fit["shape"], fit["scale_years"])

plt.figure(figsize=(8,4))
plt.plot(ages, survival)
plt.xlabel("Module age (years)")
plt.ylabel("Survival probability")
plt.title("Fitted Weibull survival curve (synthetic observations)")
plt.grid(alpha=.25)
plt.show()

years = np.arange(settings["forecast_start_year"], settings["forecast_end_year"]+1)
forecast = forecast_cohort_waste(
    cohorts, settings["base_year"], years, fit["shape"], fit["scale_years"]
)
display(forecast.head(10))

## 3. Interpret annual flows

In [ ]:
fig, ax = plt.subplots(figsize=(10,4))
ax.bar(forecast["year"], forecast["expected_module_waste_tonnes"])
ax.set(title="Expected annual module waste (synthetic demonstration)",
       xlabel="Year", ylabel="Tonnes")
plt.tight_layout()
plt.show()

forecast.to_csv(ROOT / "outputs/pv_waste_forecast_demo.csv", index=False)

## Research considerations

- Distinguish right-censoring from missing failure records.
- Separate technical retirement from collected and recycled waste.
- Add reuse/refurbishment and early-replacement pathways as explicit flow scenarios.
- Use documented Bangladesh installation cohorts and module mass data before empirical claims.
- Quantify parameter uncertainty and validate against independent observations.

The model output is conditional on the input cohort and lifetime assumptions.